# Determine major importers of Prodcom products

## Notebook README

**Related publication**

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Code author: Nicolas LIENART
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This notebooks permits to identify the key exporters to EU27 and their contribution in share to the availability of products (i.e., local production and imports). Refer to the aforementioned main manuscript and accompanying supplementary information documents for more details.

**Updates**

 - August 11, 2026: Added the file.
 - August 12, 2026: Show results for both "RER" and "RER w/o RU" regions and display all results at the bottom.

**Package versions**

 - See [`environment.yml`](./environment.yml)

**Relevant references**

- Gaulier, G., & Zignago, S. (2010). BACI: International trade database at the product-level. The 1994-2007 version (Working Papers Nos. 2010–23). CEPII. http://www.cepii.fr/CEPII/fr/publications/wp/abstract.asp?NoDoc=2726
- https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37
- https://www.cepii.fr/DATA_DOWNLOAD/baci/doc/baci_webpage.html

## Python library imports

In [1]:
import pandas as pd
import country_converter as coco

## File imports

In [3]:
# NOTE: BACI data variables
hs_year = '22'
release = '202501'
year = '2022'
hs = 'HS' + hs_year

In [4]:
# NOTE: Get BACI data here: https://www.cepii.fr/DATA_DOWNLOAD/baci/doc/baci_webpage.html
# See also: https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37

baci = pd.read_csv(f'BACI_data/BACI_HS{hs_year}_V{release}/BACI_HS{hs_year}_Y{year}_V{release}.csv', dtype={a:str for a in ['t', 'i', 'j', 'k']})
baci.rename(columns={
    'q': 'Quantity (t)',
    'v': 'Value (thousand USD)',
    't': 'Year',
    'i': 'Exporter',
    'j': 'Importer',
    'k': 'Product (HS22)',
}, inplace=True)

In [5]:
baci_country_codes = pd.read_csv(f'BACI_data/BACI_HS{hs_year}_V{release}/country_codes_V{release}.csv', dtype={'country_code': str})

In [6]:
baci_product_codes = pd.read_csv(f'BACI_data/BACI_HS{hs_year}_V{release}/product_codes_HS{hs_year}_V{release}.csv', dtype={'code': str})

In [7]:
prodcom_hotspot_product_list = pd.read_csv("output/20260811 - prodcom_data.csv", index_col=0, dtype={'Local production share of available quantity (%)': float, 'HS22 code':str})

## Find HS22 product code

In [8]:
# NOTE: Get BACI product from keyword in description (product name)
name = "ethylene"
baci_product_codes.loc[baci_product_codes['description'].str.contains(name, case=False)].sort_values(by="description", key=lambda x: x.str.len()).style.set_properties(subset=["description"], **{'text-align': 'left'})

,code,description
1287,290121,"Acyclic hydrocarbons: unsaturated, ethylene"
2093,391510,"Ethylene polymers: waste, parings and scrap"
1365,290531,"Alcohols: acyclic, diols: ethylene glycol (ethanediol)"
2993,560741,"Twine: binder or baler twine, of polyethylene or polypropylene"
2033,390190,"Ethylene polymers: in primary forms, n.e.c. in heading no. 3901"
1855,340490,"Waxes: artificial and prepared, other than of polyethylene glycol"
2031,390130,"Ethylene polymers: in primary forms, ethylene-vinyl acetate copolymers"
2101,391721,"Plastics: tubes, pipes and hoses thereof, rigid, of polymers of ethylene"
1854,340420,"Waxes: artificial and prepared, of poly(oxyethylene) (polyethylene glycol)"
1311,290322,Unsaturated chlorinated derivatives of acyclic hydrocarbons: trichloroethylene


In [9]:
# NOTE: Get BACI product from HS code
code = '390410'
baci_product_codes.loc[baci_product_codes['code'] == code]

,code,description
2043,390410,"Vinyl chloride, other halogenated olefin polym..."


## Country groups

In [10]:
cc = coco.CountryConverter()
EU27_ISO2_list = cc.EU27as('ISO2')['ISO2'].to_list()

In [11]:
continent_7_df = cc.Continent_7as('ISO2')
RER_ISO2_list = continent_7_df.loc[continent_7_df['Continent_7'] == 'Europe'].sort_values(by='ISO2')['ISO2'].to_list()

In [12]:
RER_WO_RU_ISO2_list = continent_7_df.loc[(continent_7_df['Continent_7'] == 'Europe') & (continent_7_df['ISO2'] != 'RU')].sort_values(by='ISO2')['ISO2'].to_list()

In [13]:
# NOTE: Currently RER region is defined based on country_converter's Continent7 classification.
#   See: https://github.com/IndEcol/country_converter/tree/v1.3.2#classification-schemes
# It would be prefereable to fit with Ecoinvent's definition of RER in a future version which inludes only parts of Russion in Europe.
#   See: https://geography.ecoinvent.org/#europe-and-asia

COUNTRY_CODE_GROUPS = {
    'EU27': {
        'name': 'European Union (27 members, as of 2020)',
        'list': EU27_ISO2_list
        },
    'RER': {
        'name': 'Europe (Our World in Data)',
        'list': RER_ISO2_list
        },
    'RER w/o RU': {
        'name': 'Europe (Our World in Data), without Russia',
        'list': RER_WO_RU_ISO2_list
        }
}

## Function

In [14]:
def get_importer_exporter_country(row, name_column='country_iso2'):
    exporter_country_iso2 = baci_country_codes[baci_country_codes['country_code'] == row['Exporter']][name_column]
    importer_country_iso2 = baci_country_codes[baci_country_codes['country_code'] == row['Importer']][name_column]
    return (importer_country_iso2.iloc[0], exporter_country_iso2.iloc[0])

In [15]:
def calculate_share(df, col, sum=None):
    if sum:
        sum_col = sum
    else:
        sum_col = df[col].sum()
    return df.apply(lambda row: (row[col]/sum_col), axis=1, result_type='expand')

In [16]:
def get_imports_product_code(hs22_code, country_group_code):

    country_group_list = COUNTRY_CODE_GROUPS[country_group_code]['list']

    # NOTE: Single product flows (already single year)
    baci22_product = baci[baci['Product (HS22)'] == hs22_code].copy()

    # NOTE: Add columns with country codes to facilitate use
    baci22_product[['Importer (ISO2)', 'Exporter (ISO2)']] = baci22_product.apply(get_importer_exporter_country, axis='columns', result_type='expand')

    # NOTE: Only select country that are not part of the country group but which export products to the country group
    baci22_product_group_imports = baci22_product.loc[
            (~baci22_product['Exporter (ISO2)'].isin(country_group_list)) & (baci22_product['Importer (ISO2)'].isin(country_group_list))
        ].copy()

    # NOTE: Group all flows corresponding to a certain exporter together (there are possibly many importers in the given country group)
    baci22_product_group_imports_grouped = baci22_product_group_imports.groupby(
            by=['Year', 'Exporter', 'Product (HS22)', 'Exporter (ISO2)'], as_index=False
        ).agg({
            'Value (thousand USD)': 'sum',
            'Quantity (t)': 'sum'
        })

    baci22_product_group_imports_grouped['Exporter (name)'] = baci22_product_group_imports_grouped.apply(
            lambda row: cc.convert(row['Exporter (ISO2)'], to='name_short'),
            axis=1,
            result_type='expand'
        )
    
    baci22_product_group_imports_grouped['Share'] = calculate_share(baci22_product_group_imports_grouped, 'Quantity (t)')

    return baci22_product_group_imports_grouped

## Analysis

In [19]:
# NOTE: Choose which group to use
# country_code_group = 'RER w/o RU'
# country_code_group = 'RER'
country_code_groups = ["RER", "RER w/o RU"]

# NOTE: The selected region for which we will have a look at the external exporters (countries outside of this region)
importer_group_name = 'EU27'

for index, prodcom_hotspot_product in prodcom_hotspot_product_list.iterrows():

    # NOTE: Select product by id
    
    baci_hs22_product_code = prodcom_hotspot_product['HS22 code']
    prodcom_hotspot_product_local_production_share = prodcom_hotspot_product['Local production share of available quantity (%)']
    prodcom_hotspot_product_import_share = (1 - prodcom_hotspot_product_local_production_share)
    baci_hs22_product_description = baci_product_codes.loc[baci_product_codes['code'] == baci_hs22_product_code]['description'].iloc[0]

    external_exporters = get_imports_product_code(hs22_code=baci_hs22_product_code, country_group_code=importer_group_name)

    # NOTE: Adjust share to reflect %of total availability
    external_exporters['Share'] = external_exporters['Share'] * prodcom_hotspot_product_import_share

    print(f"Id: [{prodcom_hotspot_product.name}]")
    print(f"Long name: {prodcom_hotspot_product['Long name']}")
    print(f"BACI description: {baci_hs22_product_description}")

    for country_code_group in country_code_groups:
        # NOTE: Aggregate non EU27 exporters which are part of the selected country group together
        selected_region_exporters = external_exporters['Exporter (ISO2)'].isin(COUNTRY_CODE_GROUPS[country_code_group]['list'])
        selected_group_exporters_df = external_exporters.loc[selected_region_exporters].groupby(
                by=['Year', 'Product (HS22)'], as_index=False
            ).agg({
            # 'Value (thousand USD)': 'sum',
            'Quantity (t)': 'sum',
            'Share': 'sum',
        })

        # NOTE: Add local availability share to imported share
        selected_group_exporters_df['Share'] += prodcom_hotspot_product_local_production_share 
        selected_group_exporters_df['Exporter (name)'] = COUNTRY_CODE_GROUPS[country_code_group]['name']
        selected_group_exporters_df

        # NOTE: Extract the other non EU27 exporters, filters and reorganize columns
        other_exporters_df = external_exporters.loc[~selected_region_exporters][[
                'Year',
                'Product (HS22)',
                # 'Exporter',
                'Exporter (ISO2)',
                'Exporter (name)',
                # 'Value (thousand USD)',
                'Quantity (t)',
                'Share',
            ]].copy()

        total_imports = external_exporters['Quantity (t)'].sum()
        other_exporters_df['Share'] = calculate_share(other_exporters_df, 'Quantity (t)', total_imports) * prodcom_hotspot_product_import_share

        # NOTE: Combination of non EU27 (selected group and others) exporters
        # Only show the first 5 rows
        display(pd.concat([other_exporters_df, selected_group_exporters_df]).sort_values(by='Share', ascending=False).head(5).style.format({
            'Quantity (t)': '{:.1e}'.format,
            'Share': '{:.0%}'.format,
        }))

    # if prodcom_hotspot_product.name >= 2:
    #     break

Id: [0]
Long name: Polypropylene, in primary forms
BACI description: Propylene, other olefin polymers: polypropylene in primary forms


,Year,Product (HS22),Exporter (ISO2),Exporter (name),Quantity (t),Share
0,2022,390210,nan,Europe (Our World in Data),3.3e+05,89%
51,2022,390210,SA,Saudi Arabia,5.7e+05,5%
28,2022,390210,KR,South Korea,1.5e+05,1%
74,2022,390210,EG,Egypt,1.3e+05,1%
22,2022,390210,IL,Israel,6.5e+04,1%


,Year,Product (HS22),Exporter (ISO2),Exporter (name),Quantity (t),Share
0,2022,390210,nan,"Europe (Our World in Data), without Russia",1.8e+05,88%
51,2022,390210,SA,Saudi Arabia,5.7e+05,5%
28,2022,390210,KR,South Korea,1.5e+05,1%
50,2022,390210,RU,Russia,1.5e+05,1%
74,2022,390210,EG,Egypt,1.3e+05,1%


Id: [1]
Long name: Ethylene
BACI description: Petroleum gases and other gaseous hydrocarbons: liquefied, ethylene, propylene, butylene and butadiene


,Year,Product (HS22),Exporter (ISO2),Exporter (name),Quantity (t),Share
0,2022,271114,nan,Europe (Our World in Data),4.0e+05,96%
11,2022,271114,TR,Türkiye,1.4e+05,3%
14,2022,271114,US,United States,2.6e+04,1%
1,2022,271114,CA,Canada,4.2e+03,0%
10,2022,271114,TN,Tunisia,1.6e+03,0%


...

,Year,Product (HS22),Exporter (ISO2),Exporter (name),Quantity (t),Share
0,2022,390810,nan,Europe (Our World in Data),8.1e+04,91%
48,2022,390810,US,United States,1.0e+05,4%
42,2022,390810,TR,Türkiye,3.4e+04,1%
3,2022,390810,CN,China,3.3e+04,1%
1,2022,390810,CA,Canada,2.7e+04,1%


,Year,Product (HS22),Exporter (ISO2),Exporter (name),Quantity (t),Share
0,2022,390810,nan,"Europe (Our World in Data), without Russia",4.9e+04,90%
48,2022,390810,US,United States,1.0e+05,4%
42,2022,390810,TR,Türkiye,3.4e+04,1%
3,2022,390810,CN,China,3.3e+04,1%
30,2022,390810,RU,Russia,3.2e+04,1%


Id: [51]
Long name: Polyamide -6 (PA 6)
BACI description: Polyamides: polyamide-6, -11, -12, -6,6, -6,9, -6,10 or -6,12, in primary forms


,Year,Product (HS22),Exporter (ISO2),Exporter (name),Quantity (t),Share
0,2022,390810,nan,Europe (Our World in Data),8.1e+04,91%
48,2022,390810,US,United States,1.0e+05,4%
42,2022,390810,TR,Türkiye,3.4e+04,1%
3,2022,390810,CN,China,3.3e+04,1%
1,2022,390810,CA,Canada,2.7e+04,1%


,Year,Product (HS22),Exporter (ISO2),Exporter (name),Quantity (t),Share
0,2022,390810,nan,"Europe (Our World in Data), without Russia",4.9e+04,90%
48,2022,390810,US,United States,1.0e+05,4%
42,2022,390810,TR,Türkiye,3.4e+04,1%
3,2022,390810,CN,China,3.3e+04,1%
30,2022,390810,RU,Russia,3.2e+04,1%
